# 03 -- Shadow-Mode Traffic Sampling Demo

Implements the shadow-mode router described in Chapter 4: real (production) traffic is duplicated to a
**sampled percentage** of requests, and that sample is routed to a mock agent/model client whose
output is logged and compared but **never returned to the caller**. In this course's current framing,
the "research model" client below stands in for any one of the multi-agent underwriting system's calls
being validated in shadow mode before go-live -- the isolation guarantee demonstrated here is identical
either way. The core guarantee this notebook demonstrates concretely, via fault injection, is Chapter
4's and Chapter 7's isolation rule: a shadow-path exception or timeout must never affect the primary
response path.

Entirely offline -- both "model clients" below are mock, deterministic (seeded) Python objects. No
network calls, no real API keys.

In [1]:
import random
import time
import numpy as np
import pandas as pd

RNG_SEED = 7
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

pd.set_option("display.max_colwidth", 60)
print("Environment ready. Offline, seeded, no network calls.")

Environment ready. Offline, seeded, no network calls.


## 1. Mock model clients

- `ProductionModelClient` always succeeds -- it stands in for the enterprise Azure OpenAI deployment
  that must never be affected by anything happening on the research path.
- `ResearchModelClient` is deliberately unreliable -- it raises a timeout, raises a generic error, or
  succeeds, on a random schedule -- standing in for a still-being-evaluated Claude research endpoint.
  Its failures are the whole point: they exist so the isolation guarantee below has something real to
  prove itself against.

In [2]:
class ResearchModelTimeout(Exception):
    pass


class ResearchModelError(Exception):
    pass


class ProductionModelClient:
    name = "azure-openai-gpt-enterprise"

    def generate(self, request_id):
        # Always succeeds -- this is the production path and must never be made to fail by
        # anything happening on the research side.
        return f"[PRODUCTION OUTPUT for {request_id}] narrative drafted successfully."


class ResearchModelClient:
    name = "claude-research-endpoint"

    def generate(self, request_id):
        roll = random.random()
        if roll < 0.15:
            raise ResearchModelTimeout(f"research endpoint timed out for {request_id}")
        if roll < 0.25:
            raise ResearchModelError(f"research endpoint returned a 5xx for {request_id}")
        return f"[RESEARCH OUTPUT for {request_id}] narrative drafted successfully."


production_client = ProductionModelClient()
research_client = ResearchModelClient()
print("Mock clients ready:", production_client.name, "/", research_client.name)

Mock clients ready: azure-openai-gpt-enterprise / claude-research-endpoint


## 2. The shadow-mode router

Mirrors the error-handling table in chapter 7: the production call always runs and its result is
always returned to the caller. The research call, if sampled, runs in an isolated `try`/`except`
block whose *only* possible effects are (a) a logged research-side status and (b) nothing else --
in particular, no exception raised on the research side can propagate out and affect the return
value the router hands back to the caller.

In [3]:
SAMPLING_RATE = 0.4  # matches chapter 7's point: sample a fraction, don't mirror 100% of traffic

shadow_log = []


def handle_request(request_id, sampling_rate=SAMPLING_RATE, force_sample=None):
    # 1. Production path -- always runs, always returns its result to the caller.
    production_output = production_client.generate(request_id)

    # 2. Decide whether this request is sampled into shadow mode.
    is_sampled = (random.random() < sampling_rate) if force_sample is None else force_sample

    research_status = "not_sampled"
    research_output = None

    if is_sampled:
        # 3. Research path -- isolated. Any exception here is caught HERE and never propagates.
        try:
            research_output = research_client.generate(request_id)
            research_status = "success"
        except ResearchModelTimeout:
            research_status = "timeout"
        except ResearchModelError:
            research_status = "error"
        except Exception:
            # Defensive catch-all: an unrecognized research-side failure still must not
            # escape this block, per chapter 7's fail-closed-toward-production rule.
            research_status = "unrecognized_error"

    shadow_log.append(
        {
            "request_id": request_id,
            "sampled": is_sampled,
            "research_status": research_status,
            "research_output_captured": research_output is not None,
        }
    )

    # 4. Return ONLY the production output. The research output, whatever happened to it,
    #    never reaches the caller -- exactly chapter 4's shadow-mode rule.
    return production_output


print("Shadow-mode router defined.")

Shadow-mode router defined.


## 3. Run a batch of simulated requests

Simulate 200 production requests flowing through the router. Every single one must return a valid
production response, regardless of what happened on the research side.

In [4]:
NUM_REQUESTS = 200
responses = []

for i in range(NUM_REQUESTS):
    request_id = f"REQ-{i:04d}"
    response = handle_request(request_id)
    responses.append(response)

print(f"Processed {NUM_REQUESTS} requests through the shadow-mode router.")
print(f"Collected {len(responses)} production responses.")
print(f"Collected {len(shadow_log)} shadow-mode log entries.")

Processed 200 requests through the shadow-mode router.
Collected 200 production responses.
Collected 200 shadow-mode log entries.


In [5]:
shadow_df = pd.DataFrame(shadow_log)
shadow_df["research_status"].value_counts()

research_status
not_sampled    117
success         58
timeout         14
error           11
Name: count, dtype: int64

## 4. Verify the isolation guarantee

Two assertions that must both hold, no matter how badly the research path behaved:

1. Every single request produced a valid, non-empty production response -- a research-side timeout
   or error never prevented, altered, or delayed the caller from getting a real production answer.
2. The sampling rate observed in practice is close to the configured rate, and research failures are
   visible in the log rather than silently swallowed -- the isolation guarantee is about protecting
   the *production* response, not about hiding that the research path had problems.

In [6]:
# Guarantee 1: every response is a valid production output, never affected by research failures.
assert len(responses) == NUM_REQUESTS
assert all(r.startswith("[PRODUCTION OUTPUT") for r in responses), (
    "A production response was missing or corrupted -- the isolation guarantee failed."
)
print("PASS: every request returned a valid, unaffected production response.")

# Guarantee 2: research failures are visible in the log, not silently dropped.
research_failure_statuses = {"timeout", "error", "unrecognized_error"}
observed_failures = shadow_df["research_status"].isin(research_failure_statuses).sum()
observed_sampled = shadow_df["sampled"].sum()
print(f"Sampled requests: {observed_sampled} / {NUM_REQUESTS} "
      f"({observed_sampled / NUM_REQUESTS:.1%}, target {SAMPLING_RATE:.0%})")
print(f"Research-side failures among sampled requests: {observed_failures}")
assert observed_failures > 0, "Expected at least some research-side failures given the mock client."
print("PASS: research-side failures occurred AND were captured in the log, without affecting "
      "a single production response.")

PASS: every request returned a valid, unaffected production response.
Sampled requests: 83 / 200 (41.5%, target 40%)
Research-side failures among sampled requests: 25
PASS: research-side failures occurred AND were captured in the log, without affecting a single production response.


## 5. Fault injection: force every sampled request to fail on the research side

The strongest version of the isolation test: force the research path to fail on **every** sampled
request (100% failure rate), and confirm the production path is still completely unaffected. This is
the notebook equivalent of the fault-injection test chapter 7 describes -- deliberately breaking the
research side as hard as possible and proving nothing leaks across the boundary.

In [7]:
class AlwaysFailingResearchClient:
    name = "claude-research-endpoint-FAULT-INJECTED"

    def generate(self, request_id):
        raise ResearchModelTimeout(f"forced failure for {request_id}")


# Swap in the always-failing client for this fault-injection run only.
research_client = AlwaysFailingResearchClient()
shadow_log.clear()

fault_injection_responses = [
    handle_request(f"FAULT-{i:04d}", force_sample=True) for i in range(50)
]

assert len(fault_injection_responses) == 50
assert all(r.startswith("[PRODUCTION OUTPUT") for r in fault_injection_responses)

fault_df = pd.DataFrame(shadow_log)
assert (fault_df["sampled"] == True).all()
assert (fault_df["research_status"] == "timeout").all()

print("FAULT INJECTION RESULT: 50/50 sampled requests, 50/50 forced research-side timeouts, "
      "50/50 valid production responses returned.")
print("Isolation guarantee holds even under 100% research-path failure.")

FAULT INJECTION RESULT: 50/50 sampled requests, 50/50 forced research-side timeouts, 50/50 valid production responses returned.
Isolation guarantee holds even under 100% research-path failure.


## Recap

This notebook implemented the two properties chapter 4 and chapter 7 argue a shadow-mode deployment
must have: (1) a defined sampling rate, not full traffic mirroring, and (2) a structural isolation
guarantee -- verified here under both ordinary mixed-failure conditions and an aggressive
fault-injection scenario where the research path fails 100% of the time -- that a research-model
failure can never affect the response a real caller receives. The one thing this notebook does not
attempt to simulate is the "sampler itself fails" edge case chapter 7 calls out as the one row in its
error-handling table needing the most active engineering attention; extending `handle_request` with a
fault-injected sampling-decision failure, and confirming it fails toward "skip research" rather than
toward "skip production," would be the natural next test to add.